# arcroad — Mapillary coverage probe (v0.1, thin)

**Question:** is there recent street-level imagery near the pairs we measured, so it can
show whether a median opening exists that OpenStreetMap is missing?

**Rule 5, declared before running.** Hypothesis: Hyderabad has usable imagery near our
flagged pairs. Competing explanation: any zero comes from a broken query, as on 25 Oct,
when even central London returned nothing. What separates them: control boxes with known
dense coverage are queried first with the same code; if any returns fewer than
`CONTROL_MIN_IMAGES`, the run stops and no Hyderabad number is read. Flagged and
unflagged pairs are sampled equally, so we learn whether imagery is where the detours are.

**Thresholds (rule 2), fixed in `pipeline/mapillary.py` before the run:** a pair is
covered if an image lies within 50 m of its midpoint and is under 5 years old; coverage
is useful if at least 25% of sampled flagged pairs are covered.

**Licence boundary.** Mapillary imagery is CC-BY-SA 4.0. It is used to check and correct
OpenStreetMap upstream, never imported into arcroad's ODbL data. No image is downloaded:
only ids, positions and capture times.

## 1. Code, Drive and token

The token comes from Colab's **Secrets** panel (key icon, left sidebar), name `MAPILLARY_TOKEN`, with notebook access switched on. It never appears in this notebook.

In [ ]:
REPO_URL, REPO_REF, REPO_DIR = "https://github.com/rrmlogs8896/arcroad.git", "main", "/content/arcroad"
import os, subprocess, sys, time, json, csv, datetime
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", REPO_REF], check=True)
subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"], check=False)
sys.path.insert(0, REPO_DIR)
from pipeline import mapillary as m, pairing
SHA = pairing.commit_sha(REPO_DIR)

DRIVE_BASE = "/content/drive/MyDrive/AI PROJECTS — MASTER WORKSPACE/Anthropic/Hyderabad Mobility Atlas"
POINTS_CSV = f"{DRIVE_BASE}/phase2a_detour_v08/di_points_2026-10-26_c29d9c1.csv"
OUT_DIR    = f"{DRIVE_BASE}/probe_mapillary_v01"

from google.colab import drive, userdata
drive.mount("/content/drive")
if not os.path.isdir(DRIVE_BASE):
    raise SystemExit(f"Drive folder not found: {DRIVE_BASE}")
if not os.path.exists(POINTS_CSV):
    raise SystemExit(f"Points file not found: {POINTS_CSV}")
TOKEN = userdata.get("MAPILLARY_TOKEN")
print("pipeline at", SHA, "| token loaded:", bool(TOKEN))

## 2. Controls first — if these fail, stop

In [ ]:
fetch = lambda bbox: m.fetch_images(bbox, TOKEN)
control_counts = {}
for name, bbox in m.CONTROLS.items():
    control_counts[name] = len(fetch(bbox))
    print(f"{name:20s} {control_counts[name]:5d} images")
dead = [k for k, n in control_counts.items() if n < m.CONTROL_MIN_IMAGES]
if dead:
    raise SystemExit(f"QUERY BROKEN: {dead} under {m.CONTROL_MIN_IMAGES} images. No Hyderabad result may be read.")
print("controls pass — the query works")

## 3. The verification target and the core city

In [ ]:
now_ms = time.time() * 1000
fixed = {}
for name, bbox in m.FIXED_TARGETS.items():
    imgs = fetch(bbox)
    lon, lat = (bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2
    d, age = m.nearest(imgs, lon, lat, now_ms=now_ms)
    fixed[name] = {"images_in_box": len(imgs), "nearest_m": d, "nearest_age_years": age}
    print(f"{name:34s} {len(imgs):5d} images | nearest recent "
          + ("none" if d is None else f"{d:.0f} m, {age:.1f} years old"))

## 4. Flagged vs unflagged pairs, sampled equally

In [ ]:
with open(POINTS_CSV) as f:
    records = list(csv.DictReader(f))
flagged, unflagged = m.sample_pairs(records, n_per_group=100, seed=26)
rows_f = m.coverage(flagged, fetch, now_ms=now_ms)
rows_u = m.coverage(unflagged, fetch, now_ms=now_ms)
sf, su = m.share_covered(rows_f), m.share_covered(rows_u)
status, sentence = m.verdict(control_counts, sf, su)
print(status, "—", sentence)

## 5. Save the result — positions and counts only, no images

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
stamp = datetime.date.today().isoformat()
out = f"{OUT_DIR}/mapillary_coverage_{stamp}_{(SHA or 'nosha')[:7]}.csv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["group"] + list(rows_f[0].keys()))
    w.writeheader()
    for g, rows in (("flagged", rows_f), ("unflagged", rows_u)):
        for r in rows:
            w.writerow({"group": g, **r})
meta = {"status": status, "verdict": sentence, "controls": control_counts, "fixed_targets": fixed,
        "share_flagged": sf, "share_unflagged": su, "n_per_group": 100, "seed": 26,
        "thresholds": {"CONTROL_MIN_IMAGES": m.CONTROL_MIN_IMAGES, "NEAR_M": m.NEAR_M,
                       "MAX_AGE_YEARS": m.MAX_AGE_YEARS, "USEFUL_SHARE": m.USEFUL_SHARE},
        "points_file": os.path.basename(POINTS_CSV), "commit_sha": SHA, "run_date": stamp,
        "licence_note": "Mapillary imagery CC-BY-SA 4.0; used to correct OSM upstream, not imported."}
with open(out + ".meta.json", "w") as f:
    json.dump(meta, f, indent=2, default=str)
print("saved:", out)